# Construction et entraînement des premiers modèles
### Scoring de risque de crédit — Séance 6

On repart du résultat de S5 (encodage + feature engineering). Comme cet environnement ne conserve pas les fichiers d'une session à l'autre, on refait d'abord rapidement les étapes de S4 et S5 avant d'attaquer le vrai sujet du jour : entraîner et évaluer un premier modèle.

## Étape 0 — Reconstituer le point de départ (résultat de S5)

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/Loan_default.csv")

# S4 : indicateur de coherence Age / MonthsEmployed
mois_emploi_maximum_plausible = (df["Age"] - 16) * 12
df["AgeEmploymentIncoherent"] = (df["MonthsEmployed"] > mois_emploi_maximum_plausible).astype(int)

# S5 : encodage des variables binaires
for colonne in ["HasMortgage", "HasDependents", "HasCoSigner"]:
    df[colonne] = df[colonne].map({"Yes": 1, "No": 0})

# S5 : encodage ordinal de Education
df["Education_encoded"] = df["Education"].map({"High School": 0, "Bachelor's": 1, "Master's": 2, "PhD": 3})

# S5 : One-Hot Encoding
df = pd.get_dummies(df, columns=["EmploymentType", "MaritalStatus", "LoanPurpose"],
                     prefix=["EmploymentType", "MaritalStatus", "LoanPurpose"])
colonnes_one_hot = [col for col in df.columns if col.startswith(("EmploymentType_", "MaritalStatus_", "LoanPurpose_"))]
for colonne in colonnes_one_hot:
    df[colonne] = df[colonne].astype(int)

# S5 : feature engineering
df["LoanToIncomeRatio"] = df["LoanAmount"] / df["Income"]

identifiants = df["LoanID"]
df = df.drop(columns=["LoanID", "Education"])

df.shape

(255347, 28)

---
# 1. Séparer la cible des variables explicatives

`X` contient toutes les variables qui servent à prédire. `y` contient uniquement ce qu'on cherche à prédire (`Default`). C'est une convention universelle en Machine Learning : on ne mélange jamais la cible avec les variables d'entrée, sinon le modèle "trichera" en apprenant directement la réponse.

In [2]:
X = df.drop(columns=["Default"])
y = df["Default"]

print("X :", X.shape)
print("y :", y.shape)
print()
print("Répartition de y :")
print(y.value_counts(normalize=True).round(3))

X : (255347, 27)
y : (255347,)

Répartition de y :
Default
0    0.884
1    0.116
Name: proportion, dtype: float64


# 2. Séparer entraînement et test, en respectant le déséquilibre de classes

`train_test_split` divise aléatoirement les données. On utilise `stratify=y` pour que le jeu d'entraînement et le jeu de test gardent tous les deux environ 88 % / 12 % de non-défaut/défaut — sans ça, un découpage malchanceux pourrait donner un jeu de test avec très peu de défauts, rendant l'évaluation peu fiable.

On réserve 20 % des données au test (proportion standard), et on fixe `random_state=42` pour que le découpage soit reproductible (le même à chaque exécution).

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Entraînement :", X_train.shape[0], "lignes")
print("Test          :", X_test.shape[0], "lignes")
print()
print("Proportion de défauts - train :", y_train.mean().round(3))
print("Proportion de défauts - test  :", y_test.mean().round(3))

Entraînement : 204277 lignes
Test          : 51070 lignes

Proportion de défauts - train : 0.116
Proportion de défauts - test  : 0.116


# 3. Standardiser les variables numériques

La régression logistique est sensible à l'échelle des variables. `StandardScaler` transforme chaque variable pour qu'elle ait une moyenne de 0 et un écart-type de 1 — sans changer sa distribution, juste son échelle.

**Point important : on calcule la standardisation uniquement sur le train (`fit`), puis on l'applique au test (`transform`) avec les mêmes paramètres.** Si on calculait la moyenne/écart-type sur l'ensemble des données (train + test), on laisserait "fuiter" de l'information du jeu de test vers l'entraînement — une erreur classique appelée *data leakage*, qui rendrait l'évaluation artificiellement optimiste.

In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# fit_transform sur le train : on apprend la moyenne/ecart-type ET on les applique
X_train_scaled = scaler.fit_transform(X_train)

# transform seul sur le test : on applique les MEMES parametres appris sur le train
X_test_scaled = scaler.transform(X_test)

print("Moyenne apres standardisation (train, doit etre proche de 0) :", X_train_scaled.mean().round(3))
print("Ecart-type apres standardisation (train, doit etre proche de 1) :", X_train_scaled.std().round(3))

Moyenne apres standardisation (train, doit etre proche de 0) : 0.0
Ecart-type apres standardisation (train, doit etre proche de 1) : 1.0


# 4. Premier modèle : régression logistique simple

On entraîne un premier modèle, sans aucun réglage particulier. C'est notre **référence (baseline)** : tout modèle plus complexe qu'on essaiera par la suite devra faire mieux que ça pour se justifier.

In [5]:
from sklearn.linear_model import LogisticRegression

modele_baseline = LogisticRegression(random_state=42, max_iter=1000)
modele_baseline.fit(X_train_scaled, y_train)

# Predictions sur le jeu de test
y_pred_baseline = modele_baseline.predict(X_test_scaled)
y_proba_baseline = modele_baseline.predict_proba(X_test_scaled)[:, 1]  # probabilite de defaut

print("Modele entraine.")

Modele entraine.


# 5. Évaluer avec les bonnes métriques

Rappel de S2/S3 : l'accuracy seule est trompeuse sur un dataset déséquilibré (88 %/12 %). On utilise donc les métriques identifiées dans la revue de littérature : matrice de confusion, recall, AUC-ROC.

In [6]:
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_auc_score, recall_score
)

print("--- Accuracy (a interpreter avec precaution) ---")
print(f"{accuracy_score(y_test, y_pred_baseline):.3f}")

print()
print("--- Matrice de confusion ---")
print("(lignes = realite, colonnes = prediction)")
print(confusion_matrix(y_test, y_pred_baseline))

print()
print("--- Rapport de classification (precision, recall, f1) ---")
print(classification_report(y_test, y_pred_baseline, target_names=["Non-defaut", "Defaut"]))

print()
print("--- AUC-ROC (la metrique la plus fiable ici) ---")
print(f"{roc_auc_score(y_test, y_proba_baseline):.3f}")

--- Accuracy (a interpreter avec precaution) ---
0.887

--- Matrice de confusion ---
(lignes = realite, colonnes = prediction)
[[44876   263]
 [ 5524   407]]

--- Rapport de classification (precision, recall, f1) ---
              precision    recall  f1-score   support

  Non-defaut       0.89      0.99      0.94     45139
      Defaut       0.61      0.07      0.12      5931

    accuracy                           0.89     51070
   macro avg       0.75      0.53      0.53     51070
weighted avg       0.86      0.89      0.84     51070


--- AUC-ROC (la metrique la plus fiable ici) ---
0.762


**Comment lire ces résultats :**
- L'**accuracy** est élevée mais trompeuse : elle est proche de la proportion de non-défauts (88 %), donc un modèle qui prédirait toujours "pas de défaut" aurait presque le même score.
- La **matrice de confusion** montre le vrai problème : regarde la ligne "Défaut" — combien de vrais défauts sont correctement détectés (recall) ? Avec ce modèle simple, le recall de la classe Défaut est probablement faible : le modèle a tendance à prédire "non-défaut" par défaut, précisément à cause du déséquilibre.
- L'**AUC-ROC** est la mesure la plus fiable : elle évalue la capacité du modèle à bien classer, indépendamment du déséquilibre.

# 6. Deuxième version : tenir compte du déséquilibre de classes

`class_weight='balanced'` demande au modèle de donner plus d'importance aux erreurs sur la classe minoritaire (les défauts) pendant l'entraînement, pour compenser leur rareté.

In [7]:
modele_pondere = LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced")
modele_pondere.fit(X_train_scaled, y_train)

y_pred_pondere = modele_pondere.predict(X_test_scaled)
y_proba_pondere = modele_pondere.predict_proba(X_test_scaled)[:, 1]

print("--- Matrice de confusion (modele pondere) ---")
print(confusion_matrix(y_test, y_pred_pondere))

print()
print("--- Rapport de classification (modele pondere) ---")
print(classification_report(y_test, y_pred_pondere, target_names=["Non-defaut", "Defaut"]))

print()
print("--- AUC-ROC (modele pondere) ---")
print(f"{roc_auc_score(y_test, y_proba_pondere):.3f}")

--- Matrice de confusion (modele pondere) ---
[[31134 14005]
 [ 1789  4142]]

--- Rapport de classification (modele pondere) ---
              precision    recall  f1-score   support

  Non-defaut       0.95      0.69      0.80     45139
      Defaut       0.23      0.70      0.34      5931

    accuracy                           0.69     51070
   macro avg       0.59      0.69      0.57     51070
weighted avg       0.86      0.69      0.74     51070


--- AUC-ROC (modele pondere) ---
0.762


## 7. Comparer les deux versions

In [8]:
comparaison = pd.DataFrame({
    "Modele": ["Baseline (sans ponderation)", "Pondere (class_weight=balanced)"],
    "AUC-ROC": [
        roc_auc_score(y_test, y_proba_baseline),
        roc_auc_score(y_test, y_proba_pondere),
    ],
    "Recall (classe Defaut)": [
        recall_score(y_test, y_pred_baseline),
        recall_score(y_test, y_pred_pondere),
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_baseline),
        accuracy_score(y_test, y_pred_pondere),
    ],
})
comparaison.round(3)

,Modele,AUC-ROC,Recall (classe Defaut),Accuracy
0,Baseline (sans ponderation),0.762,0.069,0.887
1,Pondere (class_weight=balanced),0.762,0.698,0.691


**Ce qu'on attend de voir :** l'AUC-ROC reste globalement similaire entre les deux versions (c'est une mesure indépendante du seuil de décision), mais le **recall de la classe Défaut** devrait nettement s'améliorer avec la pondération — le modèle détecte plus de vrais défauts, au prix probable d'un peu plus de fausses alertes (moins de precision). C'est un compromis à discuter : dans le scoring de crédit, rater un vrai défaut coûte en général plus cher qu'une fausse alerte (cf. synthèse de littérature S2).

## 8. Sauvegarder le modèle et un résumé des résultats

In [9]:
import joblib
import os

os.makedirs("../src/models", exist_ok=True)
joblib.dump(modele_pondere, "../src/models/logistic_regression_baseline.joblib")
joblib.dump(scaler, "../src/models/scaler.joblib")

comparaison.to_csv("../reports/comparaison_modeles_S6.csv", index=False)

print("Modele et resultats sauvegardes.")

Modele et resultats sauvegardes.


## Résumé

- Jeu de données séparé en entraînement (80 %) / test (20 %), avec stratification pour préserver le déséquilibre 88 %/12 % dans les deux jeux.
- Variables standardisées (fit sur le train uniquement, pour éviter le data leakage).
- Deux régressions logistiques entraînées : une simple, une pondérée pour la classe minoritaire.
- Évaluation avec AUC-ROC, matrice de confusion et recall — pas l'accuracy seule.

Prochaine étape (S7) : comparer cette référence à des méthodes d'ensemble (Random Forest, Gradient Boosting) et affiner le seuil de décision.